# Patent Emerging Radar

**Goal:** Discover and forecast emerging scientific topics from USPTO patent filings — no pre-curated topic list needed.

**Key insight:** Patents are filed 1–3 years before papers are published. A cluster of semantically-similar patents growing fast today is a leading signal of the next research wave.

## Pipeline
| Part | What happens |
|---|---|
| 0 | Setup & paths |
| 1 | Build eruption ground truth from OpenAlex (62 objective events) |
| 2 | Embed patent titles with sentence-transformers |
| 3 | Unsupervised clustering: UMAP + HDBSCAN, comparison, stability |
| 4 | Feature engineering: 18 time-series features per cluster |
| 5 | Train / Val / Test split — **test set locked** |
| 6 | LSTM + GRU architectures, train on train, evaluate on val only |
| 7 | Val set deep analysis |
| 8 | Retrospective validation: patent lead time vs OpenAlex eruptions |
| 9 | 🔒 TEST SET (locked until you say so) |
| 10 | Leaderboard: what will blow up next? |

---

## Part 0 — Setup

Run this cell at the start of every session.

In [ ]:
import json, re, pathlib, warnings, datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)

ROOT        = pathlib.Path('..').resolve()
CACHE       = ROOT / 'data' / 'raw' / 'openalex' / 'topics_cache.json'
REF_CSV     = ROOT / 'data' / 'reference' / 'openalex_topics.csv'
EMBED_DIR   = ROOT / 'data' / 'processed' / 'embeddings'
CLUSTER_DIR = ROOT / 'data' / 'processed' / 'clusters'
VAL_DIR     = ROOT / 'data' / 'processed' / 'validation'
MODEL_DIR   = ROOT / 'models'

for d in [EMBED_DIR, CLUSTER_DIR, VAL_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

print(f'Device : {DEVICE}')
print(f'Root   : {ROOT}')

---
## Part 1 — Eruption Ground Truth

We derive our validation set **purely from data**, with no hand-picking. The OpenAlex 4,516-topic quarterly panel (already cached locally) gives us objective eruption events to validate against later.

**Two types:**
- **Emerging** — tiny topic (< 500 papers/yr) that grew 5× in 5 years (unknown unknowns: XAI, Circular RNAs, IoT protocols)
- **Acceleration** — established field (< 10k papers/yr) that surged 4× in 5 years (Nobel-class: single-cell sequencing, perovskite solar, GANs)

These 62 events are the ground truth we will match patent clusters against in Part 8.

In [ ]:
print('Loading OpenAlex topics cache …')
cache   = json.loads(CACHE.read_text())
ref     = pd.read_csv(REF_CSV)

vol_slices = {k.split('::')[1]: v for k, v in cache.items()
              if k.startswith('VOL::') and isinstance(v, dict)}
qs = sorted(vol_slices)
print(f'  {len(ref)} topics | {len(vol_slices)} quarterly slices ({qs[0]} – {qs[-1]})')

In [ ]:
years = sorted({int(q[:4]) for q in vol_slices})
print(f'Building {len(ref)} × {len(years)} annual volume matrix …')

rows = []
for y in years:
    ann = defaultdict(int)
    for q in range(1, 5):
        for tid, cnt in vol_slices.get(f'{y}Q{q}', {}).items():
            ann[tid] += cnt
    for tid in ref['topic_id']:
        rows.append({'topic_id': tid, 'year': y, 'volume': ann.get(tid, 0)})

pivot = (pd.DataFrame(rows)
         .pivot(index='year', columns='topic_id', values='volume')
         .fillna(0))
print(f'Matrix shape: {pivot.shape}')

In [ ]:
# ── Eruption thresholds (adjust to widen/narrow the validation set) ──────────
GROWTH_EMERGING  = 5.0;  MAX_BASE_EMERGING  = 500;   MIN_PEAK_EMERGING  = 500
GROWTH_ACCEL     = 4.0;  MAX_BASE_ACCEL     = 10_000; MIN_PEAK_ACCEL    = 5_000
MIN_Q_DATA = 8
YR0, YR1   = 2005, 2022   # leave 2023+ as out-of-sample
ALLOWED_DOMAINS = {'Life Sciences', 'Physical Sciences', 'Health Sciences'}

NOISE_RE = re.compile(
    r'covid|pandemic|sars.cov|ukraine|pancasila|islamic|arabic language'
    r'|legal and (policy|forensic)|^diverse scientific'
    r'|^advanced technologies|^scientific research and technology$|^applied advanced',
    re.IGNORECASE)

NOBEL = {
    'T10878': (2020, 'Nobel Medicine — CRISPR'),
    'T10463': (2017, 'Nobel Physics — Gravitational waves'),
    'T10158': (2018, 'Nobel Medicine — Immune checkpoint'),
    'T11289': (2024, 'Nobel Medicine — Single-cell / spatial'),
}

ref_map = ref.set_index('topic_id')[['topic','domain']].to_dict('index')
records = []
for tid in pivot.columns:
    meta   = ref_map.get(tid, {})
    name   = meta.get('topic', tid)
    domain = meta.get('domain', '')
    if domain not in ALLOWED_DOMAINS or NOISE_RE.search(name):
        continue
    series = pivot[tid]
    for y0 in [y for y in years if YR0 <= y <= YR1-5]:
        y5 = y0 + 5
        if y5 not in series.index: continue
        v0, v5 = series[y0], series[y5]
        if v0 <= 0: continue
        if   v0 <= MAX_BASE_EMERGING and v5 >= MIN_PEAK_EMERGING and v5 >= GROWTH_EMERGING*v0:
            etype = 'emerging'
        elif MAX_BASE_EMERGING < v0 <= MAX_BASE_ACCEL and v5 >= MIN_PEAK_ACCEL and v5 >= GROWTH_ACCEL*v0:
            etype = 'acceleration'
        else: continue
        pre_q = sum(1 for yy in range(max(2000,y0-3), y0+1) for q in range(1,5) if series.get(yy,0)>0)
        if pre_q < MIN_Q_DATA: continue
        records.append(dict(topic_id=tid, topic_name=name, domain=domain,
            eruption_type=etype, base_year=y0, eruption_year=y5,
            base_volume=int(v0), peak_volume=int(v5), growth_x=round(v5/v0,1),
            is_nobel=tid in NOBEL, nobel_note=NOBEL.get(tid,('',''))[1]))
        break

gt = pd.DataFrame(records).sort_values('growth_x', ascending=False).reset_index(drop=True)
gt.to_csv(VAL_DIR / 'eruption_ground_truth.csv', index=False)
print(f'{len(gt)} eruption events  ({(gt.eruption_type=="emerging").sum()} emerging, '
      f'{(gt.eruption_type=="acceleration").sum()} acceleration, '
      f'{gt.is_nobel.sum()} Nobel-confirmed)')
gt[['topic_name','domain','eruption_type','base_year','eruption_year','growth_x']].head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cols = {'Physical Sciences':'#2196F3','Life Sciences':'#4CAF50','Health Sciences':'#FF9800'}
for dom, grp in gt.groupby('domain'):
    axes[0].scatter(grp.eruption_year, grp.growth_x, c=cols.get(dom,'grey'),
                    label=dom, alpha=0.8, s=70, zorder=3)
    nobel = grp[grp.is_nobel]
    if len(nobel):
        axes[0].scatter(nobel.eruption_year, nobel.growth_x, c='red', s=200,
                        marker='*', zorder=4, label='Nobel anchor')
axes[0].set(xlabel='Eruption year', ylabel='5-yr growth ×', title='Eruption events')
axes[0].legend(fontsize=8)
gt.groupby('eruption_year').size().plot(kind='bar', ax=axes[1], color='#5C6BC0')
axes[1].set(xlabel='Year', ylabel='Count', title='Eruptions per year')
plt.tight_layout(); plt.show()
for _, r in gt[gt.is_nobel].iterrows():
    print(f'  Nobel: {r.topic_name}  {r.base_year}→{r.eruption_year} ({r.growth_x}×)  {r.nobel_note}')

---
## Part 2 — Embed Patent Titles

`all-MiniLM-L6-v2` encodes each patent title into a 384-d semantic vector.
Output: one `.npz` per year in `data/processed/embeddings/`.

### Option A — Run on Kaggle (recommended, ~2 hrs free)
Use `notebooks/kaggle_embed.ipynb`. Upload it to kaggle.com, enable T4 GPU,
run all cells, download `embeddings.zip`, extract here:
```
data/processed/embeddings/
```
Then skip to **Part 3** — no need to run the cells below.

### Option B — Run locally (CPU or 1080 Ti)
Run the cell below. Already-completed years are skipped automatically.

| Hardware | Batch size | Per year | 1996–2025 total |
|---|---|---|---|
| CPU | 64–256 | ~2–4 hrs | ~3–4 days |
| 1080 Ti (GPU) | 512 | ~8 min | ~4 hrs |
| Kaggle T4 (free) | 512 | ~4 min | ~2 hrs |

**Kill-safe:** completed years checkpoint to disk — restart anytime and it picks up where it left off.

In [ ]:
import subprocess, sys

# ── Change these to suit your hardware and how far back you want to go ────────
EMBED_YEARS      = ('1996', '2024')   # 1996 = recommended floor (internet/genomics era)
                                       # 1990 = more LSTM training history, slightly older language
                                       # 1976 = full history (limited value pre-1990)
                                       # GPU users: '1976' '2025' for everything
EMBED_BATCH_SIZE = '64'               # CPU: 64. GPU: 512 or higher.
EMBED_THREADS    = '0'                # 0 = use all CPU cores (recommended)
# ─────────────────────────────────────────────────────────────────────────────

cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'embed_patents.py'),
    '--years',      *EMBED_YEARS,
    '--batch-size', EMBED_BATCH_SIZE,
    '--threads',    EMBED_THREADS,
]
print('Command:', ' '.join(cmd))
print('Kill and re-run at any time — completed years are skipped.\n')

result = subprocess.run(cmd, cwd=ROOT)

if result.returncode != 0:
    print('\nEmbedding failed — check errors above.')
else:
    files    = sorted(EMBED_DIR.glob('*.npz'))
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f'\n{len(files)} year files ready  ({total_mb:.0f} MB total)')
    for f in files:
        print(f'  {f.name}  {f.stat().st_size/1e6:.0f} MB')

In [ ]:
# ── Sanity check one year ──────────────────────────────────────────────────────
YEAR_CHECK = 2020
f = EMBED_DIR / f'{YEAR_CHECK}.npz'
if f.exists():
    d = np.load(f, allow_pickle=True)
    embs = d['embeddings']; titles = d['titles']
    print(f'{YEAR_CHECK}: {len(embs):,} patents  shape={embs.shape}  dtype={embs.dtype}')
    print(f'Norms (should be ~1.0): {np.linalg.norm(embs[:5], axis=1).round(4)}')
    print('Sample titles:')
    for t in titles[:5]: print(f'  {t}')
else:
    print(f'No embeddings for {YEAR_CHECK} — run Part 2 first.')

In [ ]:
# ── Verify embeddings are present (run this whether from Kaggle or local) ─────
files = sorted(EMBED_DIR.glob('*.npz'))
if not files:
    print('No embeddings found.')
    print('  → Kaggle path : extract embeddings.zip into data/processed/embeddings/')
    print('  → Local path  : run the embed cell above')
else:
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    years    = [int(f.stem) for f in files]
    print(f'Embeddings ready: {len(files)} years  ({total_mb:.0f} MB)')
    print(f'  Range : {min(years)}–{max(years)}')
    # spot check the most recent year
    d    = np.load(files[-1], allow_pickle=True)
    embs = d['embeddings']
    norms = np.linalg.norm(embs[:5], axis=1)
    print(f'  Latest: {files[-1].name}  shape={embs.shape}  norms={norms.round(3)}')
    if abs(norms.mean() - 1.0) > 0.01:
        print('  WARNING: norms not ~1.0 — embeddings may not be normalised')
    else:
        print('  Norms look good. Ready for Part 3.')

---
## Part 3 — Unsupervised Clustering

We cluster patents into semantic themes without fixing the number of clusters in advance.

**Steps:**
1. **UMAP** (384d → 10d, cosine metric) — preserves local semantic structure, makes HDBSCAN ~10× faster
2. **HDBSCAN** — density-based, finds arbitrarily-shaped clusters, marks outliers as noise (-1)
3. **k-means comparison** — show why HDBSCAN outperforms k-means on this data
4. **Rolling-window stitching** — track clusters across 2-year windows using centroid cosine similarity
5. **Stability analysis** — what % of clusters survive quarter-to-quarter?
6. **Cluster similarity heatmap** — are clusters distinct or overlapping?
7. **NMF + TF-IDF labelling** — auto-generate human-readable names

In [ ]:
# ── Option A: load pre-computed cluster results from Kaggle ───────────────────
# If you ran notebooks/kaggle_cluster.ipynb, extract cluster_output.zip here:
#   cd data/processed/clusters && unzip ~/Downloads/cluster_output.zip
# Then set CLUSTERS_FROM_KAGGLE = True to skip all clustering cells below.

import pickle

CLUSTERS_FROM_KAGGLE = False

if CLUSTERS_FROM_KAGGLE:
    cluster_panel_path = CLUSTER_DIR / 'cluster_panel.csv'
    labels_path        = CLUSTER_DIR / 'cluster_labels.csv'
    centroids_path     = CLUSTER_DIR / 'cluster_centroids.pkl'
    titles_path        = CLUSTER_DIR / 'cluster_titles.pkl'

    missing = [p for p in [cluster_panel_path, labels_path, centroids_path, titles_path]
               if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f'Missing files: {missing}\n'
            'Extract cluster_output.zip into data/processed/clusters/ first.'
        )

    panel_full        = pd.read_csv(cluster_panel_path)
    labels_df         = pd.read_csv(labels_path)
    cluster_labels    = dict(zip(labels_df.cluster_id, labels_df.auto_label))

    with open(centroids_path, 'rb') as _f:
        cluster_centroids = pickle.load(_f)
    with open(titles_path, 'rb') as _f:
        cluster_titles = pickle.load(_f)

    print(f'Loaded from Kaggle output:')
    print(f'  {panel_full.cluster_id.nunique()} clusters')
    print(f'  {panel_full.date.nunique()} quarters')
    print('Skip to Part 4.')
else:
    print('CLUSTERS_FROM_KAGGLE = False — running clustering locally (cells below).')


In [ ]:
try:
    import umap as umap_lib
    import hdbscan as hdbscan_lib
    print('UMAP and HDBSCAN ready.')
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','umap-learn','hdbscan','-q'])
    import umap as umap_lib, hdbscan as hdbscan_lib
    print('Installed UMAP and HDBSCAN.')

In [ ]:
# ── Clustering config — tune these to trade off granularity vs. stability ─────
WINDOW_YEARS   = 2    # rolling window width
UMAP_NEIGHBORS = 50   # larger = more global structure
UMAP_DIMS      = 10   # UMAP output dims for HDBSCAN
HDBSCAN_MIN    = 50   # min patents per cluster (lower = more, smaller clusters)
CENTROID_SIM   = 0.85 # cosine sim threshold to stitch cluster across windows
CLUSTER_YEARS  = list(range(2005, 2025))
MIN_ACTIVE_Q   = 16   # drop clusters with fewer active quarters

def load_year(year):
    f = EMBED_DIR / f'{year}.npz'
    if not f.exists(): return None
    d = np.load(f, allow_pickle=True)
    return d['ids'], d['dates'], d['titles'], d['embeddings']

def cosine_sim(a, b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9))

def get_quarter(date_str):
    y, m = int(date_str[:4]), int(date_str[5:7])
    return y, (m-1)//3 + 1

available = [y for y in CLUSTER_YEARS if (EMBED_DIR/f'{y}.npz').exists()]
print(f'{len(available)} years ready: {available[:3]} … {available[-3:]}')

In [ ]:
# ── Cluster one rolling window ────────────────────────────────────────────────
def cluster_window(yrs, viz_pts=2000):
    parts = [p for p in (load_year(y) for y in yrs) if p is not None]
    if not parts: return None
    ids    = np.concatenate([p[0] for p in parts])
    dates  = np.concatenate([p[1] for p in parts])
    titles = np.concatenate([p[2] for p in parts])
    embs   = np.concatenate([p[3] for p in parts])

    reducer = umap_lib.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIMS,
                            metric='cosine', random_state=42, low_memory=True)
    reduced = reducer.fit_transform(embs)

    clusterer = hdbscan_lib.HDBSCAN(min_cluster_size=HDBSCAN_MIN,
                                    metric='euclidean', cluster_selection_method='eom')
    labels = clusterer.fit_predict(reduced)
    n_cl   = len(set(labels)) - (1 if -1 in labels else 0)
    noise  = (labels==-1).mean()*100

    centroids = {}
    for c in set(labels):
        if c == -1: continue
        mask = labels==c
        cent = embs[mask].mean(axis=0)
        centroids[c] = cent / (np.linalg.norm(cent)+1e-9)

    # 2D UMAP for visualisation (fit only on a sample to save time)
    viz_embs = embs[:viz_pts]
    reducer2d = umap_lib.UMAP(n_neighbors=30, n_components=2,
                              metric='cosine', random_state=42, low_memory=True)
    reduced2d = reducer2d.fit_transform(viz_embs)

    print(f'  {yrs}: {len(embs):,} patents → {n_cl} clusters  noise={noise:.1f}%')
    return dict(ids=ids, dates=dates, titles=titles, labels=labels,
                centroids=centroids, n_clusters=n_cl,
                reduced2d=reduced2d, labels2d=labels[:viz_pts])

if len(available) >= WINDOW_YEARS:
    _w = available[:WINDOW_YEARS]
    print(f'Dry-run window: {_w}')
    _res = cluster_window(_w)
else:
    print('Need ≥2 years of embeddings — run Part 2 first.')

In [ ]:
# ── UMAP 2D visualisation ─────────────────────────────────────────────────────
if '_res' in dir() and _res:
    pts, lbs = _res['reduced2d'], _res['labels2d']
    n_show   = min(_res['n_clusters'], 20)
    cmap     = cm.get_cmap('tab20', n_show)
    fig, ax  = plt.subplots(figsize=(9,6))
    noise_m  = lbs==-1
    ax.scatter(pts[noise_m,0], pts[noise_m,1], c='#cccccc', s=1, alpha=0.25, label='noise')
    for c in range(n_show):
        m = lbs==c
        if m.sum()==0: continue
        ax.scatter(pts[m,0], pts[m,1], s=2, color=cmap(c), alpha=0.7)
    ax.set(title=f'Patent UMAP — window {_w}  ({_res["n_clusters"]} clusters, first 2000 pts)',
           xlabel='UMAP-1', ylabel='UMAP-2')
    plt.tight_layout(); plt.show()

In [ ]:
# ── k-means vs HDBSCAN comparison ─────────────────────────────────────────────
# k-means forces every point into a cluster and assumes spherical clusters.
# On patent embeddings the clusters are irregular and there's genuine noise,
# so HDBSCAN should win on silhouette score.
from sklearn.cluster import KMeans

if '_res' in dir() and _res:
    # Sample 5000 points for speed
    parts = [p for p in (load_year(y) for y in _w) if p is not None]
    embs_sample = np.concatenate([p[3] for p in parts])[:5000]

    reduced_sample = umap_lib.UMAP(n_neighbors=30, n_components=5,
                                   metric='cosine', random_state=42).fit_transform(embs_sample)

    ks      = [10, 20, 30, 50]
    km_sil  = []
    for k in ks:
        lbl = KMeans(n_clusters=k, random_state=42, n_init='auto').fit_predict(reduced_sample)
        km_sil.append(silhouette_score(reduced_sample, lbl, sample_size=2000))

    hdb_lbl = hdbscan_lib.HDBSCAN(min_cluster_size=HDBSCAN_MIN).fit_predict(reduced_sample)
    valid   = hdb_lbl != -1
    hdb_sil = silhouette_score(reduced_sample[valid], hdb_lbl[valid], sample_size=min(2000,valid.sum()))

    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(ks, km_sil, 'o-', label='k-means', color='#FF7043')
    ax.axhline(hdb_sil, ls='--', color='#43A047', label=f'HDBSCAN ({hdb_sil:.3f})')
    ax.set(xlabel='k (k-means) or fixed line (HDBSCAN)', ylabel='Silhouette score',
           title='HDBSCAN vs k-means — silhouette on 5,000-patent sample')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f'Best k-means silhouette: {max(km_sil):.3f}  |  HDBSCAN: {hdb_sil:.3f}')

In [ ]:
# ── Full rolling-window run + cross-window stitching ──────────────────────────
cluster_series    = defaultdict(lambda: defaultdict(int))   # gid → {(y,q): count}
cluster_centroids = {}   # gid → latest 384-d centroid
cluster_titles    = defaultdict(list)   # gid → sample titles (for labelling)
stability_log     = []   # track how many clusters stitch vs. born each window
next_gid          = 0

windows = [available[i:i+WINDOW_YEARS] for i in range(len(available)-WINDOW_YEARS+1)]
print(f'Processing {len(windows)} windows …\n')

for window in windows:
    res = cluster_window(window)
    if res is None: continue
    stitched, born = 0, 0
    local_to_global = {}
    for loc, cent in res['centroids'].items():
        best_gid, best_s = -1, 0.0
        for gid, gcent in cluster_centroids.items():
            s = cosine_sim(cent, gcent)
            if s > best_s: best_s, best_gid = s, gid
        if best_s >= CENTROID_SIM:
            local_to_global[loc] = best_gid
            cluster_centroids[best_gid] = cent
            stitched += 1
        else:
            local_to_global[loc] = next_gid
            cluster_centroids[next_gid] = cent
            next_gid += 1; born += 1
    stability_log.append({'window': str(window), 'stitched': stitched,
                          'born': born, 'total': res['n_clusters']})
    for pid, date, title, lbl in zip(res['ids'], res['dates'], res['titles'], res['labels']):
        if lbl==-1 or lbl not in local_to_global: continue
        gid = local_to_global[lbl]
        cluster_series[gid][get_quarter(date)] += 1
        if len(cluster_titles[gid]) < 25:
            cluster_titles[gid].append(str(title))

print(f'Total global clusters: {next_gid}')
print(f'With ≥{MIN_ACTIVE_Q} quarters: {sum(1 for s in cluster_series.values() if len(s)>=MIN_ACTIVE_Q)}')

In [ ]:
# ── Cluster stability chart ───────────────────────────────────────────────────
# Shows what fraction of clusters each window are continuations vs. new births.
if stability_log:
    stab = pd.DataFrame(stability_log)
    fig, ax = plt.subplots(figsize=(12,4))
    x = range(len(stab))
    ax.bar(x, stab['stitched'], label='Stitched (continuing)', color='#43A047')
    ax.bar(x, stab['born'],     bottom=stab['stitched'], label='Born (new)', color='#FF7043')
    ax.set_xticks(list(x)[::2])
    ax.set_xticklabels(stab['window'].iloc[::2], rotation=45, fontsize=7)
    ax.set(ylabel='Cluster count', title='Cluster stability across rolling windows')
    ax.legend(); plt.tight_layout(); plt.show()
    avg_stitch = stab['stitched'].sum() / stab['total'].sum() * 100
    print(f'Average stitch rate: {avg_stitch:.1f}%  (higher = more stable cluster identities)')

In [ ]:
# ── Build cluster panel (dense quarterly counts) ──────────────────────────────
panel_rows = []
for gid, ts in cluster_series.items():
    if len(ts) < MIN_ACTIVE_Q: continue
    for (y,q), cnt in ts.items():
        panel_rows.append({'cluster_id':gid,'year':y,'quarter':q,
                           'date':f'{y}-{(q-1)*3+1:02d}-01','count':cnt})

panel = pd.DataFrame(panel_rows)
all_yq = sorted({(r.year, r.quarter) for _, r in panel.iterrows()})
full_rows = []
for gid in panel.cluster_id.unique():
    sub = panel[panel.cluster_id==gid].set_index(['year','quarter'])['count']
    for y,q in all_yq:
        full_rows.append({'cluster_id':gid,'year':y,'quarter':q,
                          'date':f'{y}-{(q-1)*3+1:02d}-01','count':sub.get((y,q),0)})

panel_full = pd.DataFrame(full_rows).sort_values(['cluster_id','date']).reset_index(drop=True)
panel_full.to_csv(CLUSTER_DIR/'cluster_panel.csv', index=False)
print(f'Panel: {panel_full.cluster_id.nunique()} clusters × {len(all_yq)} quarters')
panel_full.head(6)

In [ ]:
# ── Auto-label clusters with TF-IDF + NMF ────────────────────────────────────
from sklearn.decomposition import NMF

cluster_labels = {}
active_ids = set(panel_full.cluster_id.unique())

for gid in active_ids:
    titles_list = cluster_titles.get(gid, [])
    if not titles_list:
        cluster_labels[gid] = f'cluster_{gid}'; continue
    corpus = [' '.join(titles_list)]
    try:
        vec = TfidfVectorizer(max_features=40, stop_words='english', ngram_range=(1,2))
        X   = vec.fit_transform(corpus)
        # NMF with 1 component = top theme
        nmf = NMF(n_components=1, random_state=42).fit(X)
        top_idx = nmf.components_[0].argsort()[::-1][:5]
        terms   = [vec.get_feature_names_out()[i] for i in top_idx]
        cluster_labels[gid] = ', '.join(terms)
    except Exception:
        cluster_labels[gid] = f'cluster_{gid}'

print('Sample auto-labels (NMF + TF-IDF):')
for gid in list(active_ids)[:10]:
    n = panel_full[panel_full.cluster_id==gid]['count'].sum()
    print(f'  C{gid:03d}  {n:6,} patents   {cluster_labels[gid]}')

labels_df = pd.DataFrame([{'cluster_id':k,'auto_label':v} for k,v in cluster_labels.items()])
labels_df.to_csv(CLUSTER_DIR/'cluster_labels.csv', index=False)

In [ ]:
# ── Cluster similarity heatmap (top 30 clusters by total count) ───────────────
top30_ids = (panel_full.groupby('cluster_id')['count'].sum()
             .nlargest(30).index.tolist())
avail_ids = [gid for gid in top30_ids if gid in cluster_centroids]

if len(avail_ids) >= 5:
    mat = np.array([cluster_centroids[gid] for gid in avail_ids])
    # Cosine sim matrix
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
    mat_n = mat / norms
    sim   = mat_n @ mat_n.T

    fig, ax = plt.subplots(figsize=(10,8))
    im = ax.imshow(sim, cmap='RdYlGn', vmin=0.5, vmax=1.0)
    labels_short = [cluster_labels.get(g, f'C{g}')[:20] for g in avail_ids]
    ax.set_xticks(range(len(avail_ids))); ax.set_xticklabels(labels_short, rotation=90, fontsize=7)
    ax.set_yticks(range(len(avail_ids))); ax.set_yticklabels(labels_short, fontsize=7)
    plt.colorbar(im, ax=ax, label='Cosine similarity')
    ax.set_title('Top-30 cluster similarity matrix — off-diagonal should be < 0.85')
    plt.tight_layout(); plt.show()
    print(f'Mean off-diagonal sim: {(sim.sum()-np.trace(sim))/(len(avail_ids)**2-len(avail_ids)):.3f}')

---
## Part 4 — Feature Engineering

We build **18 features** per cluster per quarter. These fall into 5 groups:

| Group | Features | What they capture |
|---|---|---|
| Core signal | `log_count`, `count_share` | Absolute size + relative share of global patents |
| Derivatives | `velocity`, `acceleration`, `jerk` | Rate of change at 3 orders |
| Momentum | `mom_4q`, `mom_8q` | Year-over-year and 2-year growth |
| Rolling stats | `roll_mean_4q/8q`, `roll_std_4q/8q`, `above_trend`, `macd`, `z_score` | Trend, volatility, deviation |
| Structural | `log_cumsum`, `sin_q`, `cos_q` | Maturity, seasonality |

In [ ]:
def make_features(panel: pd.DataFrame) -> pd.DataFrame:
    p = panel.copy().sort_values(['cluster_id','date'])

    # ── Core ──────────────────────────────────────────────────────────────────
    p['log_count'] = np.log1p(p['count'])

    # global patents per quarter (denominator for share)
    global_q = p.groupby('date')['count'].transform('sum').replace(0, np.nan)
    p['count_share'] = p['count'] / global_q

    # ── Derivatives (grouped by cluster) ──────────────────────────────────────
    grp = p.groupby('cluster_id')['log_count']
    p['velocity']     = grp.diff().fillna(0)
    p['acceleration'] = p.groupby('cluster_id')['velocity'].diff().fillna(0)
    p['jerk']         = p.groupby('cluster_id')['acceleration'].diff().fillna(0)

    # ── Momentum ──────────────────────────────────────────────────────────────
    p['mom_4q'] = p.groupby('cluster_id')['log_count'].diff(4).fillna(0)
    p['mom_8q'] = p.groupby('cluster_id')['log_count'].diff(8).fillna(0)

    # ── Rolling statistics ─────────────────────────────────────────────────────
    def roll(series, w, fn):
        return series.groupby(p['cluster_id']).transform(
            lambda x: getattr(x.rolling(w, min_periods=1), fn)()
        )
    p['roll_mean_4q'] = roll(p['log_count'], 4, 'mean')
    p['roll_std_4q']  = roll(p['log_count'], 4, 'std').fillna(0)
    p['roll_mean_8q'] = roll(p['log_count'], 8, 'mean')
    p['roll_std_8q']  = roll(p['log_count'], 8, 'std').fillna(0)

    # ── Trend deviation ────────────────────────────────────────────────────────
    p['above_trend'] = p['log_count'] - p['roll_mean_8q']
    p['macd']        = p['roll_mean_4q'] - p['roll_mean_8q']  # short vs long MA
    p['z_score']     = p['above_trend'] / (p['roll_std_8q'] + 1e-6)

    # ── Relative rank ──────────────────────────────────────────────────────────
    p['global_rank_pctl'] = p.groupby('date')['log_count'].rank(pct=True)

    # ── Structural ────────────────────────────────────────────────────────────
    p['log_cumsum'] = np.log1p(
        p.groupby('cluster_id')['count'].cumsum()
    )
    p['sin_q'] = np.sin(2 * np.pi * p['quarter'] / 4)
    p['cos_q'] = np.cos(2 * np.pi * p['quarter'] / 4)

    p = p.fillna(0)
    return p

ALL_FEATURES = [
    'log_count', 'count_share',
    'velocity', 'acceleration', 'jerk',
    'mom_4q', 'mom_8q',
    'roll_mean_4q', 'roll_std_4q', 'roll_mean_8q', 'roll_std_8q',
    'above_trend', 'macd', 'z_score',
    'global_rank_pctl',
    'log_cumsum',
    'sin_q', 'cos_q',
]
TARGET = 'log_count'

if 'panel_full' not in dir() or len(panel_full) == 0:
    panel_full = pd.read_csv(CLUSTER_DIR/'cluster_panel.csv')

df = make_features(panel_full)
print(f'Features: {len(ALL_FEATURES)}')
print(f'Clusters: {df.cluster_id.nunique()}  |  Rows: {len(df)}')
df[ALL_FEATURES].describe().round(3)

In [ ]:
# ── Feature correlation matrix ────────────────────────────────────────────────
corr = df[ALL_FEATURES].corr()
fig, ax = plt.subplots(figsize=(12,10))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(ALL_FEATURES))); ax.set_xticklabels(ALL_FEATURES, rotation=90, fontsize=8)
ax.set_yticks(range(len(ALL_FEATURES))); ax.set_yticklabels(ALL_FEATURES, fontsize=8)
ax.set_title('Feature correlation matrix')
plt.tight_layout(); plt.show()
# Highly correlated pairs (|r| > 0.9)
high_corr = [(ALL_FEATURES[i], ALL_FEATURES[j], round(corr.iloc[i,j],3))
             for i in range(len(ALL_FEATURES)) for j in range(i+1, len(ALL_FEATURES))
             if abs(corr.iloc[i,j]) > 0.9]
if high_corr:
    print('Highly correlated pairs (|r|>0.9) — consider dropping one:')
    for a,b,r in high_corr: print(f'  {a} × {b} = {r}')

In [ ]:
# ── Feature importance via gradient boosting (train-era data only) ────────────
# Quick proxy importance — use to decide if any features are useless.
# We use the TRAIN period here (2005–2017) to avoid any leakage.
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance

train_mask = (df['year'] < 2018)
X_fi = df.loc[train_mask, ALL_FEATURES].values
y_fi = df.loc[train_mask, TARGET].values

# Sample for speed (up to 20k rows)
rng = np.random.default_rng(42)
if len(X_fi) > 20_000:
    idx = rng.choice(len(X_fi), 20_000, replace=False)
    X_fi, y_fi = X_fi[idx], y_fi[idx]

print('Fitting gradient boosting for feature importance (train era only) …')
gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
gb.fit(X_fi, y_fi)

pi = permutation_importance(gb, X_fi, y_fi, n_repeats=5, random_state=42)
importances = pd.Series(pi.importances_mean, index=ALL_FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8,6))
importances.plot(kind='barh', ax=ax, color='#5C6BC0')
ax.set(title='Feature permutation importance (gradient boosting, train era)',
       xlabel='Mean importance decrease')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
print('Top 5 features:')
print(importances.head(5).to_string())

---
## Part 5 — Train / Val / Test Split

**Boundaries** (based on anchor-point quarter — the last quarter of the input window):

| Split | Anchor range | Purpose |
|---|---|---|
| **Train** | 2005 Q1 → 2017 Q4 | Fit model weights |
| **Val** | 2018 Q1 → 2020 Q4 | Tune hyperparameters, compare architectures |
| **Test** | 2021 Q1 → 2023 Q4 | **ONE-TIME final evaluation** — locked below |
| **Future** | 2024 Q1 → present | Leaderboard inference only |

**Why anchor-point splitting?** The split is on the *last input quarter*, not the *target quarter* (which is HORIZON quarters later). This prevents any look-ahead leakage: the model never sees future data in its input window during training.

### 🔒 TEST SET GATE
The cell below defines `RUN_TEST_SET = False`. **Do not change it** until you have finished all hyperparameter tuning on the validation set. Once you peek at test performance, you can never un-peek — every decision you make afterward is contaminated.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  TEST SET GATE                                                       ║
# ║  Set to True ONLY when all val-set tuning is complete.                   ║
# ║  You get ONE shot at the test set — make it count.                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
RUN_TEST_SET = False

if RUN_TEST_SET:
    print('⚠️  WARNING: test set is UNLOCKED. Make sure you are done tuning on val.')
else:
    print('✅ Test set locked. All model evaluation below uses VALIDATION set only.')
    print('   Change RUN_TEST_SET = True in this cell when ready for final evaluation.')

In [ ]:
# ── Visual split timeline ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 2))
zones = [
    (2005, 2018, '#A5D6A7', 'Train\n2005–2017'),
    (2018, 2021, '#90CAF9', 'Val\n2018–2020'),
    (2021, 2024, '#FFCC80', 'Test 🔒\n2021–2023'),
    (2024, 2026, '#EF9A9A', 'Future\n(leaderboard)'),
]
for x0, x1, col, lbl in zones:
    ax.barh(0, x1-x0, left=x0, height=0.5, color=col, edgecolor='white', linewidth=2)
    ax.text((x0+x1)/2, 0, lbl, ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(2004, 2027); ax.set_ylim(-0.5, 0.5)
ax.set_yticks([]); ax.set_xlabel('Year')
ax.set_title('Temporal split — anchor-point based (no leakage)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Forecasting config ────────────────────────────────────────────────────────
WINDOW   = 8            # lookback quarters (2 years)
HORIZONS = [8, 12]      # model horizons: 2yr, 3yr
BATCH    = 32
EPOCHS   = 80
LR       = 3e-4

TRAIN_END = (2017, 4)
VAL_END   = (2020, 4)
TEST_END  = {8: (2023, 4), 12: (2022, 4)}  # per-horizon: data_end - H quarters

PRIMARY_H = HORIZONS[0]  # 2yr — used for val analysis and retrospective

print(f'Window={WINDOW}q  Horizons={[f"{h//4}yr({h}q)" for h in HORIZONS]}')
print(f'Train <= {TRAIN_END}  |  Val <= {VAL_END}  |  Test <= {TEST_END}')


In [ ]:
# ── Build sliding-window samples ─────────────────────────────────────────────
def build_samples(df, split, horizon):
    """Anchor-point split. y = log-growth delta over horizon quarters."""
    Xs, ys, metas = [], [], []
    for cid, grp in df.groupby('cluster_id'):
        grp  = grp.sort_values('date').reset_index(drop=True)
        vals = grp[ALL_FEATURES].values
        n    = len(vals)
        for i in range(WINDOW, n - horizon + 1):
            anchor_row = grp.iloc[i - 1]
            ay, aq     = int(anchor_row['year']), int(anchor_row['quarter'])
            in_train   = (ay, aq) <= TRAIN_END
            in_val     = (not in_train) and (ay, aq) <= VAL_END
            in_test    = not in_train and not in_val and (ay, aq) <= TEST_END[horizon]
            if split == 'train' and not in_train: continue
            if split == 'val'   and not in_val:   continue
            if split == 'test'  and not in_test:  continue
            Xs.append(vals[i - WINDOW:i])
            ys.append(
                float(grp.iloc[i + horizon - 1][TARGET]) - float(grp.iloc[i - 1][TARGET])
            )
            metas.append({'cluster_id': cid, 'anchor_year': ay, 'anchor_q': aq})
    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.float32),
            metas)

X_train, y_train, meta_train = build_samples(df, 'train', PRIMARY_H)
X_val,   y_val,   meta_val   = build_samples(df, 'val',   PRIMARY_H)
X_test,  y_test,  meta_test  = build_samples(df, 'test',  PRIMARY_H)

print(f'Primary ({PRIMARY_H//4}yr) — Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'y_train  mean={y_train.mean():.3f}  std={y_train.std():.3f}  (near-0 mean expected)')
print(f'y_val    mean={y_val.mean():.3f}  std={y_val.std():.3f}')
print('(Test stats hidden until gate is unlocked)')


In [ ]:
# ── Verify no temporal leakage ────────────────────────────────────────────────
train_anchors = {(m['anchor_year'], m['anchor_q']) for m in meta_train}
val_anchors   = {(m['anchor_year'], m['anchor_q']) for m in meta_val}
test_anchors  = {(m['anchor_year'], m['anchor_q']) for m in meta_test}

overlap_tv = train_anchors & val_anchors
overlap_vt = val_anchors   & test_anchors
overlap_tr = train_anchors & test_anchors

assert len(overlap_tv) == 0, f'Train/val overlap: {overlap_tv}'
assert len(overlap_vt) == 0, f'Val/test overlap: {overlap_vt}'
assert len(overlap_tr) == 0, f'Train/test overlap: {overlap_tr}'

print('✅  No temporal overlap between any splits.')
print(f'   Train anchors: {min(train_anchors)} → {max(train_anchors)}')
print(f'   Val anchors  : {min(val_anchors)} → {max(val_anchors)}')
print(f'   Test anchors : {min(test_anchors)} → {max(test_anchors)}')

In [ ]:
# ── Normalize — fit scaler on TRAIN only ────────────────────────────────────
import joblib
T, F = WINDOW, len(ALL_FEATURES)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train.reshape(-1, F)).reshape(-1, T, F)
X_val_sc   = scaler.transform(X_val.reshape(-1, F)).reshape(-1, T, F)
X_test_sc  = scaler.transform(X_test.reshape(-1, F)).reshape(-1, T, F)
joblib.dump(scaler, MODEL_DIR / f'scaler_{PRIMARY_H//4}yr.pkl')
print(f'Scaler fitted on primary ({PRIMARY_H//4}yr) TRAIN, saved.')


---
## Part 6 — LSTM and GRU Models

We train both architectures with identical hyperparameters and compare on the **validation set only**. The better model gets used for the leaderboard and retrospective validation.

**Architecture overview:**
```
GRU:  Input(8, 18) → GRU(64, seq=True) → Dropout(0.2) → GRU(32) → Dropout(0.2) → Dense(1)
LSTM: Input(8, 18) → LSTM(64, seq=True) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(1)
```

**Loss:** Huber (delta=1.0) — behaves like MSE near zero, like MAE for large errors. Robust to the occasional patent-count spike.

**Primary metric:** Spearman rank correlation — we care about ranking clusters correctly, not exact count values.

In [ ]:
# ── Option A: load models from kaggle_pipeline.ipynb output ─────────────────
# Extract pipeline_output.zip, then set True to skip training cells below.
#   unzip ~/Downloads/pipeline_output.zip -d data/processed/
#   mv data/processed/models models/   (if needed)

MODELS_FROM_KAGGLE = False

if MODELS_FROM_KAGGLE:
    from tensorflow import keras as _keras
    horizon_models = {}
    for H in HORIZONS:
        hlabel  = f'{H//4}yr'
        sc_path = MODEL_DIR / f'scaler_{hlabel}.pkl'
        if not sc_path.exists():
            raise FileNotFoundError(f'Missing {sc_path} — extract pipeline_output.zip first.')
        sc_h  = joblib.load(sc_path)
        gru_h = _keras.models.load_model(sorted(MODEL_DIR.glob(f'gru_{hlabel}_*.keras'))[-1])
        lstm_h= _keras.models.load_model(sorted(MODEL_DIR.glob(f'lstm_{hlabel}_*.keras'))[-1])
        X_v, y_v, _ = build_samples(df, 'val',  H)
        X_te,y_te,_ = build_samples(df, 'test', H)
        X_v_sc  = sc_h.transform(X_v.reshape(-1, F)).reshape(-1, T, F)
        X_te_sc = sc_h.transform(X_te.reshape(-1,F)).reshape(-1, T, F)
        gru_sp  = spearmanr(y_v, gru_h.predict(X_v_sc,  verbose=0).flatten())[0]
        lstm_sp = spearmanr(y_v, lstm_h.predict(X_v_sc, verbose=0).flatten())[0]
        best_h  = gru_h if gru_sp >= lstm_sp else lstm_h
        horizon_models[H] = {
            'gru': gru_h, 'lstm': lstm_h, 'best': best_h,
            'best_arch': 'GRU' if gru_sp >= lstm_sp else 'LSTM',
            'scaler': sc_h,
            'X_val': X_v,   'y_val': y_v,   'X_val_sc': X_v_sc,
            'X_test': X_te, 'y_test': y_te, 'X_test_sc': X_te_sc,
            'val_sp_gru': gru_sp, 'val_sp_lstm': lstm_sp,
        }
        print(f'{hlabel}: GRU rho={gru_sp:.4f}  LSTM rho={lstm_sp:.4f}  best={horizon_models[H]["best_arch"]}')
    _ph = horizon_models[PRIMARY_H]
    gru_model  = _ph['gru'];  lstm_model = _ph['lstm']
    best_model = _ph['best']; best_name  = _ph['best_arch']
    X_val_sc   = _ph['X_val_sc'];  y_val  = _ph['y_val']
    X_test_sc  = _ph['X_test_sc']; y_test = _ph['y_test']
    scaler     = _ph['scaler']
    print('Models loaded. Skip to Part 7.')
else:
    print('MODELS_FROM_KAGGLE = False — training locally (cells below).')


In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f'TensorFlow {tf.__version__}  |  Keras {keras.__version__}')
print(f'GPUs visible: {tf.config.list_physical_devices("GPU")}')

In [ ]:
def build_model(arch: str, window=WINDOW, n_feat=F, lr=LR):
    inp = keras.Input(shape=(window, n_feat), name='input')
    RNN = keras.layers.GRU if arch == 'gru' else keras.layers.LSTM
    x   = RNN(64, return_sequences=True, name=f'{arch}_1')(inp)
    x   = keras.layers.Dropout(0.2)(x)
    x   = RNN(32, name=f'{arch}_2')(x)
    x   = keras.layers.Dropout(0.2)(x)
    out = keras.layers.Dense(1, name='output')(x)
    m   = keras.Model(inp, out, name=arch)
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.Huber(delta=1.0),
              metrics=['mae'])
    return m

build_model('gru').summary()


In [ ]:
def train_model(model, name, X_tr, y_tr, X_v, y_v):
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    ckpt  = str(MODEL_DIR / f'{name}_{stamp}.keras')
    cbs   = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=5, verbose=0),
        keras.callbacks.ModelCheckpoint(ckpt, save_best_only=True, verbose=0),
    ]
    hist = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                     epochs=EPOCHS, batch_size=BATCH, callbacks=cbs, verbose=1)
    return hist

if not MODELS_FROM_KAGGLE:
    horizon_models = {}
    for H in HORIZONS:
        hlabel = f'{H//4}yr'
        print(f'\n{"="*50}\nHorizon {hlabel} ({H}Q)\n{"="*50}')
        X_tr_h, y_tr_h, _ = build_samples(df, 'train', H)
        X_v_h,  y_v_h,  _ = build_samples(df, 'val',   H)
        X_te_h, y_te_h, _ = build_samples(df, 'test',  H)
        sc_h = StandardScaler()
        X_tr_sc_h = sc_h.fit_transform(X_tr_h.reshape(-1, F)).reshape(-1, T, F)
        X_v_sc_h  = sc_h.transform(X_v_h.reshape(-1, F)).reshape(-1, T, F)
        X_te_sc_h = sc_h.transform(X_te_h.reshape(-1,F)).reshape(-1, T, F)
        joblib.dump(sc_h, MODEL_DIR / f'scaler_{hlabel}.pkl')
        print(f'  train: {len(y_tr_h):,}  val: {len(y_v_h):,}  test: {len(y_te_h):,}')
        gru_h  = build_model('gru')
        print(f'  Training GRU ...')
        hist_gru_h  = train_model(gru_h,  f'gru_{hlabel}',  X_tr_sc_h, y_tr_h, X_v_sc_h, y_v_h)
        lstm_h = build_model('lstm')
        print(f'  Training LSTM ...')
        hist_lstm_h = train_model(lstm_h, f'lstm_{hlabel}', X_tr_sc_h, y_tr_h, X_v_sc_h, y_v_h)
        gru_sp  = spearmanr(y_v_h, gru_h.predict(X_v_sc_h,  verbose=0).flatten())[0]
        lstm_sp = spearmanr(y_v_h, lstm_h.predict(X_v_sc_h, verbose=0).flatten())[0]
        best_h  = gru_h if gru_sp >= lstm_sp else lstm_h
        horizon_models[H] = {
            'gru': gru_h, 'lstm': lstm_h, 'best': best_h,
            'best_arch': 'GRU' if gru_sp >= lstm_sp else 'LSTM',
            'scaler': sc_h,
            'X_val': X_v_h,   'y_val': y_v_h,   'X_val_sc': X_v_sc_h,
            'X_test': X_te_h, 'y_test': y_te_h, 'X_test_sc': X_te_sc_h,
            'val_sp_gru': gru_sp, 'val_sp_lstm': lstm_sp,
            'hist_gru': hist_gru_h, 'hist_lstm': hist_lstm_h,
        }
        print(f'  Best: {horizon_models[H]["best_arch"]}  val Spearman={max(gru_sp,lstm_sp):.4f}')
    # Expose primary-horizon vars for downstream val-analysis cells
    _ph = horizon_models[PRIMARY_H]
    gru_model  = _ph['gru'];  lstm_model = _ph['lstm']
    best_model = _ph['best']; best_name  = _ph['best_arch']
    X_val_sc   = _ph['X_val_sc'];  y_val  = _ph['y_val']
    X_test_sc  = _ph['X_test_sc']; y_test = _ph['y_test']
    scaler     = _ph['scaler']


In [ ]:
# ── Training summary ─────────────────────────────────────────────────────────
summary = []
for H, hd in horizon_models.items():
    summary.append({
        'Horizon': f'{H//4}yr ({H}Q)',
        'GRU val rho':  round(hd['val_sp_gru'],  4),
        'LSTM val rho': round(hd['val_sp_lstm'], 4),
        'Best':         hd['best_arch'],
    })
print(pd.DataFrame(summary).to_string(index=False))


In [ ]:
# ── Training curves (primary horizon) ───────────────────────────────────────
if 'hist_gru' in horizon_models.get(PRIMARY_H, {}):
    _hg = horizon_models[PRIMARY_H]['hist_gru']
    _hl = horizon_models[PRIMARY_H]['hist_lstm']
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    for col, (hist, name) in enumerate([(_hg, 'GRU'), (_hl, 'LSTM')]):
        axes[0, col].plot(hist.history['loss'],     label='train')
        axes[0, col].plot(hist.history['val_loss'], label='val')
        axes[0, col].set(title=f'{name} — Huber loss ({PRIMARY_H//4}yr)', xlabel='Epoch')
        axes[0, col].legend()
        axes[1, col].plot(hist.history['mae'],     label='train')
        axes[1, col].plot(hist.history['val_mae'], label='val')
        axes[1, col].set(title=f'{name} — MAE ({PRIMARY_H//4}yr)', xlabel='Epoch')
        axes[1, col].legend()
    plt.tight_layout(); plt.show()
else:
    print('(Training curves not available — models loaded from Kaggle)')


In [ ]:
# ── Validation metrics + baselines (all horizons) ────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error

log_count_idx = ALL_FEATURES.index('log_count')
steps_arr     = np.arange(WINDOW, dtype=np.float32)
A_fit         = np.vstack([steps_arr, np.ones(WINDOW)]).T

for H, hd in horizon_models.items():
    y_v      = hd['y_val']
    log_hist = hd['X_val'][:, :, log_count_idx]
    slopes   = np.linalg.lstsq(A_fit, log_hist.T, rcond=None)[0][0]
    y_persist = np.zeros_like(y_v)
    y_trend   = slopes * H
    gru_pred  = hd['gru'].predict(hd['X_val_sc'],  verbose=0).flatten()
    lstm_pred = hd['lstm'].predict(hd['X_val_sc'], verbose=0).flatten()
    rows = []
    for pred, name in [(gru_pred, 'GRU'), (lstm_pred, 'LSTM'),
                       (y_trend, 'Linear trend'), (y_persist, 'Persistence (0)')]:
        sp, _ = spearmanr(y_v, pred)
        rmse  = float(np.sqrt(mean_squared_error(y_v, pred)))
        mae   = float(mean_absolute_error(y_v, pred))
        rows.append({'Model': name, 'Spearman': round(sp,4),
                     'RMSE': round(rmse,4), 'MAE': round(mae,4)})
    print(f'\n── {H//4}yr ({H}Q) val metrics ──')
    print(pd.DataFrame(rows).to_string(index=False))

# Expose primary-horizon best_pred for downstream scatter plots
_ph = horizon_models[PRIMARY_H]
best_pred = _ph['best'].predict(_ph['X_val_sc'], verbose=0).flatten()
gru_m  = {'Spearman': _ph['val_sp_gru'],
          'pred': _ph['gru'].predict(_ph['X_val_sc'], verbose=0).flatten()}
lstm_m = {'Spearman': _ph['val_sp_lstm'],
          'pred': _ph['lstm'].predict(_ph['X_val_sc'], verbose=0).flatten()}
print(f'\nPrimary ({PRIMARY_H//4}yr) best: {best_name}')


---
## Part 7 — Validation Set Deep Analysis

Before touching the test set, thoroughly understand where the model succeeds and fails on the val set. This is where you iterate.

In [ ]:
# ── Predicted vs actual scatter (val set) ─────────────────────────────────────
sp, _ = spearmanr(y_val, best_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_val, best_pred, alpha=0.25, s=8, c='#5C6BC0')
lims = [min(y_val.min(), best_pred.min()), max(y_val.max(), best_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set(xlabel='Actual log-count', ylabel='Predicted',
            title=f'{best_name} val — Spearman ρ={sp:.3f}')

residuals = best_pred - y_val
axes[1].hist(residuals, bins=40, color='#5C6BC0', edgecolor='white')
axes[1].axvline(0, color='red', ls='--')
axes[1].axvline(residuals.mean(), color='orange', ls='--',
                label=f'mean={residuals.mean():.3f}')
axes[1].set(title='Residual distribution (val)', xlabel='Predicted − Actual')
axes[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Error by growth quartile (are we better at slow or fast-growing clusters?) ─
val_df = pd.DataFrame({'actual':y_val, 'pred':best_pred,
                        'abs_err': np.abs(best_pred-y_val),
                        **{k:v for k,v in zip(['cid','ay','aq'],
                           zip(*[(m['cluster_id'],m['anchor_year'],m['anchor_q'])
                                 for m in meta_val]))}}).reset_index(drop=True)
val_df['growth_quartile'] = pd.qcut(val_df['actual'], 4,
                                     labels=['Q1 slow','Q2','Q3','Q4 fast'])
print('Mean absolute error by growth quartile:')
print(val_df.groupby('growth_quartile')['abs_err'].mean().round(4).to_string())

In [ ]:
# ── Top 5 best and worst predicted clusters on val ────────────────────────────
if 'cluster_labels' not in dir() or not cluster_labels:
    _ld = pd.read_csv(CLUSTER_DIR/'cluster_labels.csv')
    cluster_labels = dict(zip(_ld.cluster_id, _ld.auto_label))

val_agg = (val_df.groupby('cid')
           .agg(mean_err=('abs_err','mean'), n=('abs_err','count'))
           .query('n >= 3')
           .sort_values('mean_err'))

print('Best predicted clusters (lowest mean val abs-error):')
for cid, row in val_agg.head(5).iterrows():
    print(f'  C{int(cid):03d}  err={row.mean_err:.4f}  n={int(row.n)}  label: {cluster_labels.get(int(cid),"?")[:50]}')
print('\nWorst predicted clusters:')
for cid, row in val_agg.tail(5).iterrows():
    print(f'  C{int(cid):03d}  err={row.mean_err:.4f}  n={int(row.n)}  label: {cluster_labels.get(int(cid),"?")[:50]}')

In [ ]:
# ── Tuning guidance ───────────────────────────────────────────────────────────
sp_val = max(gru_m['Spearman'], lstm_m['Spearman'])
print(f'Val Spearman: {sp_val:.4f}')
if sp_val < 0.3:
    print('  → Weak. Check: (1) feature quality, (2) is the panel dense enough, '
          '(3) try adding rolling_std features, (4) increase WINDOW to 12.')
elif sp_val < 0.5:
    print('  → Moderate. Try: (1) increase GRU/LSTM units to 128, '
          '(2) add a 3rd recurrent layer, (3) reduce Dropout to 0.1.')
elif sp_val < 0.7:
    print('  → Good. Fine-tune: (1) try Bidirectional wrapper, '
          '(2) add attention head on top of GRU, (3) tune HORIZON.')
else:
    print('  → Strong. Ready to consider unlocking the test set.')

---
## Part 8 — Retrospective Validation

The key question: **how many quarters before a topic erupted in academic publications did the corresponding patent cluster start growing?**

We match each of the 62 OpenAlex eruption events to the nearest patent cluster (by keyword overlap in auto-labels), then compute the lead time. A positive lead time confirms the patent signal precedes the publication signal.

> All data used here is from the val-era or earlier — no test data needed.

In [ ]:
gt = pd.read_csv(VAL_DIR/'eruption_ground_truth.csv')
print(f'Ground truth: {len(gt)} eruption events')

In [ ]:
def match_topic_to_cluster(topic_name, labels, topn=3):
    qwords = set(re.sub(r'[^a-z ]','',topic_name.lower()).split()) - \
             {'and','of','in','the','for','a','an','with','using','based'}
    scores = []
    for cid, lbl in labels.items():
        lwords = set(re.sub(r'[^a-z ]','',lbl.lower()).split())
        ov = len(qwords & lwords)
        if ov > 0: scores.append((cid, ov))
    return sorted(scores, key=lambda x:-x[1])[:topn]

GROWTH_THR = 0.3   # velocity threshold to declare cluster 'taking off'

def cluster_takeoff(cid, panel_df):
    sub = panel_df[panel_df.cluster_id==cid].sort_values('date')
    if len(sub) < 8: return None
    sub = make_features(sub.copy())
    for _, row in sub.iterrows():
        if row.get('velocity', 0) >= GROWTH_THR:
            return int(row['year'])
    return None

results = []
for _, ev in gt.iterrows():
    matches = match_topic_to_cluster(ev['topic_name'], cluster_labels)
    if not matches:
        results.append({**ev.to_dict(), 'matched_cluster':None,
                        'matched_label':None, 'takeoff_year':None, 'lead_quarters':None})
        continue
    best_cid = matches[0][0]
    takeoff  = cluster_takeoff(best_cid, panel_full)
    lead     = (ev['eruption_year'] - takeoff)*4 if takeoff else None
    results.append({**ev.to_dict(), 'matched_cluster':best_cid,
                    'matched_label':cluster_labels.get(best_cid,''),
                    'takeoff_year':takeoff, 'lead_quarters':lead})

results_df = pd.DataFrame(results)
matched    = results_df[results_df.lead_quarters.notna()]
print(f'Matched {len(matched)}/{len(gt)} events to clusters')
if len(matched):
    print(f'Mean lead  : {matched.lead_quarters.mean():.1f} quarters ({matched.lead_quarters.mean()/4:.1f} yrs)')
    print(f'Median lead: {matched.lead_quarters.median():.1f} quarters')
    print(f'Patent led (positive): {(matched.lead_quarters>0).sum()}/{len(matched)}')

In [ ]:
if len(matched):
    fig, axes = plt.subplots(1, 2, figsize=(13,5))
    axes[0].hist(matched.lead_quarters, bins=15, color='#5C6BC0', edgecolor='white')
    axes[0].axvline(0, color='red', ls='--', label='no lead')
    if len(matched):
        axes[0].axvline(matched.lead_quarters.mean(), color='orange', ls='--',
                        label=f'mean={matched.lead_quarters.mean():.1f}q')
    axes[0].set(xlabel='Lead quarters', ylabel='Count',
                title='Patent cluster → publication eruption lead time')
    axes[0].legend()

    sc = axes[1].scatter(matched.growth_x, matched.lead_quarters,
                         c=matched.eruption_year, cmap='viridis', s=70, alpha=0.8)
    plt.colorbar(sc, ax=axes[1], label='Eruption year')
    axes[1].axhline(0, color='red', ls='--', alpha=0.5)
    axes[1].set(xlabel='Growth ×', ylabel='Lead quarters',
                title='Does bigger eruption = longer patent lead?')
    plt.tight_layout(); plt.show()

    nobel_rows = results_df[results_df.is_nobel & results_df.lead_quarters.notna()]
    if len(nobel_rows):
        print('Nobel anchor lead times:')
        for _, r in nobel_rows.iterrows():
            print(f'  {r.topic_name}  lead={r.lead_quarters:.0f}q  ({r.nobel_note})')

In [ ]:
results_df.to_csv(VAL_DIR/'validation_results.csv', index=False)
print(f'Saved → {VAL_DIR}/validation_results.csv')
results_df[['topic_name','eruption_type','eruption_year','growth_x',
            'matched_label','takeoff_year','lead_quarters']].head(15)

---
## Part 9 — 🔒 TEST SET EVALUATION

> **Do not run these cells until you have completely finished all hyperparameter tuning on the validation set.**
>
> Once you look at test performance, every decision you make afterward is subtly biased by it. You only get one honest test set evaluation.

To unlock: go back to the gate cell in **Part 5** and set `RUN_TEST_SET = True`, then re-run that cell, then come back here.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔒  TEST SET GATE — THIS CELL WILL ERROR UNTIL YOU UNLOCK IT
# ══════════════════════════════════════════════════════════════════════════════
if not RUN_TEST_SET:
    raise RuntimeError(
        "\n"
        "╔══════════════════════════════════════════════════════════════╗\n"
        "║  TEST SET IS LOCKED                                          ║\n"
        "║                                                              ║\n"
        "║  Set RUN_TEST_SET = True in Part 5 ONLY after you have       ║\n"
        "║  finished ALL hyperparameter tuning on the validation set.   ║\n"
        "║                                                              ║\n"
        "║  Peeking at test performance contaminates all future         ║\n"
        "║  validation comparisons. You get ONE shot.                   ║\n"
        "╚══════════════════════════════════════════════════════════════╝"
    )
print('🔓 Test set unlocked. Proceeding with final evaluation.')

In [ ]:
# ── Final test set metrics ────────────────────────────────────────────────────
if RUN_TEST_SET:
    test_pred = best_model.predict(X_test_sc, verbose=0).flatten()
    t_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    t_mae  = mean_absolute_error(y_test, test_pred)
    t_sp,_ = spearmanr(y_test, test_pred)
    t_smape= np.mean(2*np.abs(test_pred-y_test)/(np.abs(test_pred)+np.abs(y_test)+1e-8))*100

    summary = pd.DataFrame([
        {'Split':'Val ('+best_name+')', 'RMSE':comp.loc[comp.Model==best_name,'RMSE'].values[0],
         'MAE':comp.loc[comp.Model==best_name,'MAE'].values[0],
         'sMAPE':comp.loc[comp.Model==best_name,'sMAPE'].values[0],
         'Spearman':comp.loc[comp.Model==best_name,'Spearman'].values[0]},
        {'Split':'Test ('+best_name+')', 'RMSE':round(t_rmse,4), 'MAE':round(t_mae,4),
         'sMAPE':round(t_smape,2), 'Spearman':round(t_sp,4)},
    ])
    print('── FINAL RESULTS ──────────────────────────────────────')
    print(summary.to_string(index=False))
    print()
    if abs(t_sp - max(gru_m['Spearman'],lstm_m['Spearman'])) > 0.1:
        print('⚠️  Val/test Spearman gap > 0.1 — possible overfitting to val during tuning.')

In [ ]:
# ── Test predicted vs actual scatter ─────────────────────────────────────────
if RUN_TEST_SET:
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(y_test, test_pred, alpha=0.25, s=8, c='#E53935')
    lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set(xlabel='Actual', ylabel='Predicted',
           title=f'Test set — {best_name}  Spearman ρ={t_sp:.3f}')
    plt.tight_layout(); plt.show()

---
## Part 10 — Leaderboard: What Will Blow Up Next?

Using the trained model to forecast 2 years forward for every current cluster. This does **not** use test data — it uses the most recent available quarters as input and predicts beyond the data horizon.

In [ ]:
# ── Multi-horizon forward inference ─────────────────────────────────────────
leaderboards = {}
A_fit_lb = np.vstack([np.arange(WINDOW, dtype=np.float32), np.ones(WINDOW)]).T

for H, hd in horizon_models.items():
    hlabel = f'{H//4}yr'
    rows   = []
    for cid, grp in df.groupby('cluster_id'):
        g = grp.sort_values('date')
        if len(g) < WINDOW: continue
        recent    = g.tail(WINDOW)[ALL_FEATURES].values.astype(np.float32)
        recent_sc = hd['scaler'].transform(recent).reshape(1, T, F)
        pred_delta = float(hd['best'].predict(recent_sc, verbose=0)[0][0])
        last_log   = float(g.iloc[-1]['log_count'])
        rows.append({
            'cluster_id':         cid,
            'auto_label':         cluster_labels.get(cid, f'cluster_{cid}'),
            'last_log_count':     round(last_log, 3),
            'forecast_log_count': round(last_log + pred_delta, 3),
            'predicted_growth':   round(pred_delta, 3),
            'last_date':          str(g.iloc[-1]['date']),
        })
    lb = (pd.DataFrame(rows).sort_values('predicted_growth', ascending=False)
          .reset_index(drop=True))
    lb.to_csv(CLUSTER_DIR / f'leaderboard_{hlabel}.csv', index=False)
    leaderboards[H] = lb
    print(f'{hlabel} top 5:')
    print(lb[['auto_label','predicted_growth']].head(5).to_string(index=False))
    print()

# 5yr linear extrapolation
extrap_rows = []
for cid, grp in df.groupby('cluster_id'):
    g = grp.sort_values('date')
    if len(g) < WINDOW: continue
    log_hist = g.tail(WINDOW)['log_count'].values.astype(np.float32)
    slope    = np.linalg.lstsq(A_fit_lb, log_hist, rcond=None)[0][0]
    last_log = float(g.iloc[-1]['log_count'])
    extrap_rows.append({
        'cluster_id':              cid,
        'auto_label':              cluster_labels.get(cid, f'cluster_{cid}'),
        'last_log_count':          round(last_log, 3),
        'extrapolated_growth_5yr': round(float(slope * 20), 3),
        'last_date':               str(g.iloc[-1]['date']),
    })
lb_5yr = (pd.DataFrame(extrap_rows)
          .sort_values('extrapolated_growth_5yr', ascending=False)
          .reset_index(drop=True))
lb_5yr.to_csv(CLUSTER_DIR / 'leaderboard_5yr_extrap.csv', index=False)
leaderboards['5yr_extrap'] = lb_5yr
print('5yr extrapolation (linear, not a model) top 5:')
print(lb_5yr[['auto_label','extrapolated_growth_5yr']].head(5).to_string(index=False))


In [ ]:
# ── Multi-horizon leaderboard bar chart ──────────────────────────────────────
lb2 = leaderboards[8]
lb3 = leaderboards[12]
lb5 = leaderboards['5yr_extrap']
top15_ids = lb2.head(15)['cluster_id'].tolist()

def get_growth(lb, ids, col='predicted_growth'):
    m = lb.set_index('cluster_id')[col]
    return [m.get(cid, 0) for cid in ids]

g2 = get_growth(lb2, top15_ids)
g3 = get_growth(lb3, top15_ids)
g5 = get_growth(lb5, top15_ids, 'extrapolated_growth_5yr')
labels_short = [cluster_labels.get(cid, f'C{cid}')[:45] for cid in top15_ids]

x = np.arange(len(top15_ids))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(x + w, g2[::-1], w, label='2yr model',   color='#5C6BC0')
ax.barh(x,     g3[::-1], w, label='3yr model',   color='#42A5F5')
ax.barh(x - w, g5[::-1], w, label='5yr extrap',  color='#B0BEC5', hatch='//')
ax.set_yticks(x)
ax.set_yticklabels(labels_short[::-1], fontsize=8)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.legend()
ax.set(xlabel='Predicted log-count growth',
       title='Top 15 patent clusters — 2yr, 3yr model + 5yr linear extrapolation')
plt.tight_layout(); plt.show()


In [ ]:
# ── Top 5 cluster trajectories + forecast markers ────────────────────────────
top5 = leaderboards[PRIMARY_H].head(5)['cluster_id'].tolist()
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)

horizon_styles = [
    (8,  '#5C6BC0', '2yr'),
    (12, '#42A5F5', '3yr'),
    ('5yr_extrap', '#90A4AE', '5yr↗'),
]

for i, cid in enumerate(top5):
    sub   = df[df.cluster_id == cid].sort_values('date')
    dates = pd.to_datetime(sub['date'])
    vals  = sub['log_count'].values
    ax    = axes[i]
    ax.plot(dates, vals, lw=2, color='#5C6BC0')
    ax.fill_between(dates, vals, alpha=0.1, color='#5C6BC0')
    last_dt  = dates.iloc[-1]
    last_val = vals[-1]
    for H_key, col, lbl in horizon_styles:
        lb_h = leaderboards[H_key]
        col_name = 'predicted_growth' if H_key != '5yr_extrap' else 'extrapolated_growth_5yr'
        row  = lb_h[lb_h.cluster_id == cid]
        if len(row) == 0: continue
        delta   = row.iloc[0][col_name]
        months  = (H_key if isinstance(H_key, int) else 20) * 3
        fut_dt  = last_dt + pd.DateOffset(months=months)
        marker  = 'x' if H_key == '5yr_extrap' else 'o'
        ax.plot([last_dt, fut_dt], [last_val, last_val + delta],
                '--', color=col, lw=1.2, label=lbl)
        ax.scatter([fut_dt], [last_val + delta], color=col, s=40, marker=marker, zorder=5)
    ax.set_title(cluster_labels.get(cid, f'C{cid}')[:28], fontsize=8)
    ax.set_xlabel('Year'); ax.tick_params(axis='x', rotation=45)
    if i == 0: ax.set_ylabel('log(count+1)'); ax.legend(fontsize=7)

plt.suptitle('Top-5 clusters — history + 2yr/3yr model + 5yr linear extrapolation', y=1.02)
plt.tight_layout(); plt.show()


---
## Summary

| Part | Key output |
|---|---|
| 1 | `validation/eruption_ground_truth.csv` — 62 objective eruption events |
| 2 | `embeddings/YYYY.npz` — 384-d patent embeddings per year |
| 3 | `clusters/cluster_panel.csv`, `cluster_labels.csv` — quarterly cluster counts + NMF labels |
| 4 | 18-feature engineered DataFrame `df` |
| 5 | `feature_scaler.pkl` — scaler fitted on train only |
| 6 | `models/gru_*.keras`, `models/lstm_*.keras` — trained checkpoints |
| 8 | `validation/validation_results.csv` — lead-time analysis |
| 9 | 🔒 Test metrics (after unlocking) |
| 10 | `clusters/leaderboard.csv` — forward-looking ranked clusters |

### Headline result template
> *"Across Y matched eruption events, the corresponding patent cluster showed sustained growth a mean of **X quarters** before the OpenAlex publication eruption. [Nobel anchor]: single-cell sequencing cluster took off in [year], [N] quarters before the Nobel-confirmed eruption."*

### Tuning checklist (before unlocking test set)
- [ ] Val Spearman ≥ 0.5
- [ ] Residuals approximately zero-mean (no systematic bias)
- [ ] Error not dominated by a single cluster type
- [ ] LSTM vs GRU comparison done — best model chosen
- [ ] Feature importance reviewed — no dominant noise features